# R3 · Chạy toàn bộ công việc thực nghiệm Role 3 (`run_seq2seq_en.py` & `analyze.py`)

Notebook này bao trọn toàn bộ đầu ra của **R3 (Seq2seq tiếng Anh & Phân tích/Biểu đồ)** theo `Plan4project.pdf`:
1. Kiểm tra unit test trên CPU (`tests/test_run_seq2seq_en.py`, `tests/test_analyze.py`).
2. Xuất đủ **4 file baseline "không sửa"** (`prediction = source`) cho cả tiếng Việt và tiếng Anh:
   - `results/predictions/baseline_vi_light.jsonl`
   - `results/predictions/baseline_vi_heavy.jsonl`
   - `results/predictions/baseline_en_light.jsonl`
   - `results/predictions/baseline_en_heavy.jsonl`
3. Chạy `grammarly/coedit-large` trên `data/en_dev.jsonl` (Tuần 1) và `data/en_test.jsonl` × `{light, heavy}` (Tuần 2):
   - `results/predictions/coedit-large_en_light.jsonl`
   - `results/predictions/coedit-large_en_heavy.jsonl`
4. Chạy `src/analyze.py` sinh toàn bộ biểu đồ **RQ1–RQ4** (`.png` + `.pdf`), bảng thống kê **5 nhãn lỗi OCR** và danh sách **ví dụ lỗi điển hình** (`results/figures/`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, pathlib, sys

if pathlib.Path("src/run_seq2seq_en.py").exists():
    REPO_DIR = pathlib.Path(".").resolve()
elif pathlib.Path("../src/run_seq2seq_en.py").exists():
    REPO_DIR = pathlib.Path("..").resolve()
else:
    REPO_DIR = pathlib.Path("fin-ocrfix").resolve()

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
assert pathlib.Path("src/run_seq2seq_en.py").exists(), "Không tìm thấy src/run_seq2seq_en.py"

In [ ]:
# 1. Chạy toàn bộ unit test CPU trước khi chạy thực nghiệm
!python -m unittest discover -s tests -v

## 2. Xuất toàn bộ file Baseline "không sửa" (VI & EN, Light & Heavy)
Không cần GPU — chạy ngay khi R1 đẩy `data/vi_test.jsonl` và `data/en_test.jsonl`.

In [ ]:
test_files = [p for p in ("data/vi_test.jsonl", "data/en_test.jsonl") if pathlib.Path(p).exists()]
if not test_files:
    print("[Bỏ qua] Chưa có data/vi_test.jsonl hoặc data/en_test.jsonl từ R1.")
else:
    joined = " ".join(test_files)
    cmd = f"{sys.executable} src/run_seq2seq_en.py --baseline --data {joined}"
    print(">>>", cmd)
    assert os.system(cmd) == 0

## 3. Chạy thử `grammarly/coedit-large` trên `data/en_dev.jsonl` (Tuần 1, ngày 3–7)

In [ ]:
MODEL_ID = "grammarly/coedit-large"
PREFIX = "Fix grammatical errors in this sentence: "
BATCH_SIZE = 8
DTYPE = "auto"  # auto -> bfloat16 trên Ampere+, float32 trên T4

if pathlib.Path("data/en_dev.jsonl").exists():
    cmd = (
        f'{sys.executable} src/run_seq2seq_en.py '
        f'--model {MODEL_ID} --data data/en_dev.jsonl '
        f'--prefix "{PREFIX}" '
        f'--out_dir results/predictions_dev --log_dir results/logs_dev '
        f'--batch_size {BATCH_SIZE} --dtype {DTYPE}'
    )
    print(">>>", cmd)
    assert os.system(cmd) == 0
else:
    print("[Bỏ qua] Chưa có data/en_dev.jsonl từ R1.")

## 4. Chạy FULL `grammarly/coedit-large` trên `data/en_test.jsonl` (Tuần 2, ngày 1–4)
Tự tách thành `results/predictions/coedit-large_en_light.jsonl` và `coedit-large_en_heavy.jsonl`, có hỗ trợ resume.

In [ ]:
if pathlib.Path("data/en_test.jsonl").exists():
    cmd = (
        f'{sys.executable} src/run_seq2seq_en.py '
        f'--model {MODEL_ID} --data data/en_test.jsonl '
        f'--prefix "{PREFIX}" '
        f'--batch_size {BATCH_SIZE} --dtype {DTYPE}'
    )
    print(">>>", cmd)
    assert os.system(cmd) == 0
else:
    print("[Bỏ qua] Chưa có data/en_test.jsonl từ R1.")

## 5. Sinh toàn bộ biểu đồ RQ1–RQ4, thống kê 5 nhãn lỗi & trích xuất ví dụ điển hình (Tuần 2, ngày 5–7)
Chạy sau khi R5 đã cập nhật `results/summary.csv` (hoặc chạy sớm trực tiếp trên các file đã có trong `results/predictions/`).

In [ ]:
has_summary = pathlib.Path("results/summary.csv").exists()
has_preds = pathlib.Path("results/predictions").exists() and any(pathlib.Path("results/predictions").glob("*.jsonl"))

if has_summary or has_preds:
    cmd = (
        f"{sys.executable} src/analyze.py "
        f"--summary results/summary.csv "
        f"--pred_dir results/predictions/ "
        f"--out results/figures/"
    )
    print(">>>", cmd)
    assert os.system(cmd) == 0
else:
    print("[Bỏ qua] Chưa có results/summary.csv hoặc results/predictions/*.jsonl.")

In [ ]:
# Kiểm tra các file log của R3
import json, pandas as pd

logs = sorted(pathlib.Path("results/logs").glob("*.meta.json"))
r3_logs = [p for p in logs if "coedit" in p.name or "baseline" in p.name]
if r3_logs:
    df_log = pd.DataFrame([json.loads(p.read_text(encoding="utf-8")) for p in r3_logs])
    display(df_log[["model_id", "lang", "noise", "total_lines", "empty_prediction_count", "prefix_echo_count", "sec_per_line", "output_path"]])
else:
    print("Chưa có file log R3 trong results/logs/.")